# Multi-Label Classification

Train and use multi-label classifier for 14 CheXpert disease labels.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
from sklearn.metrics import roc_auc_score, classification_report

## Multi-Label Classifier Model

In [ ]:
class MultiLabelClassifier(nn.Module):
    def __init__(self, input_dim=512, hidden_dim=256, num_classes=14):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 1024),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )
    
    def forward(self, x):
        return self.net(x)

## CheXpert Labels

In [ ]:
CHEXPERT_LABELS = [
    'Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema',
    'Enlarged Cardiomediastinum', 'Fracture', 'Lung Lesion',
    'Lung Opacity', 'Pleural Effusion', 'Pleural Other',
    'Pneumonia', 'Pneumothorax', 'Support Devices', 'No Finding'
]

## Load Trained Classifier

In [ ]:
def load_classifier(checkpoint_path='models/multilabel_classifier_biomedclip_best.pt', device='cpu'):
    """Load trained classifier."""
    model = MultiLabelClassifier()
    
    checkpoint = torch.load(checkpoint_path, map_location=device)
    if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint:
        model.load_state_dict(checkpoint['model_state_dict'])
    else:
        model.load_state_dict(checkpoint)
    
    model.eval()
    model.to(device)
    return model

def predict(model, embedding, device='cpu'):
    """Predict disease probabilities."""
    embedding = embedding.to(device)
    with torch.no_grad():
        logits = model(embedding)
        probs = torch.sigmoid(logits)
    return probs.cpu()

## Initialize Classifier

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
classifier = load_classifier(device=device)
print(f"Classifier loaded on {device}")

## Test Prediction

In [ ]:
# Load sample embedding
data = torch.load('embeddings/mimic_image_embeddings.pt')
sample_emb = data['embeddings'][0:1]

# Predict
probs = predict(classifier, sample_emb, device=device)

# Display results
disease_probs = {label: float(prob) for label, prob in zip(CHEXPERT_LABELS, probs[0])}
top_diseases = sorted(disease_probs.items(), key=lambda x: x[1], reverse=True)[:5]

print("Top 5 Predicted Diseases:")
for disease, prob in top_diseases:
    print(f"  {disease}: {prob:.3f}")

## Batch Prediction

In [ ]:
def predict_batch(model, embeddings, device='cpu', batch_size=64):
    """Predict for multiple embeddings."""
    model.eval()
    all_probs = []
    
    for i in range(0, len(embeddings), batch_size):
        batch = embeddings[i:i+batch_size].to(device)
        with torch.no_grad():
            logits = model(batch)
            probs = torch.sigmoid(logits)
        all_probs.append(probs.cpu())
    
    return torch.cat(all_probs, dim=0)

## Evaluation (if you have test set with labels)

In [ ]:
# Uncomment to evaluate on test set
# test_embeddings = torch.load('embeddings/test_embeddings.pt')['embeddings']
# test_labels = torch.load('embeddings/test_labels.pt')

# predictions = predict_batch(classifier, test_embeddings, device=device)

# # Compute AUC for each label
# for i, label in enumerate(CHEXPERT_LABELS):
#     auc = roc_auc_score(test_labels[:, i], predictions[:, i])
#     print(f"{label}: AUC = {auc:.3f}")